Let's dive into the data and start the exploratory data analysis (EDA). Based on the project structure
Those Python scripts in the `modeling` directory (`train.py`, `predict.py`, etc.) are likely set up for the main machine learning pipeline. For exploratory analysis, the best approach is usually to use a Jupyter Notebook, which allows you to inspect the data interactively.

Here’s a great way you could get started:

1.  **Create a New Notebook**: In the `notebooks/` directory of the project, you can create a new notebook (e.g., `EDA.ipynb`). This is the perfect place for this kind of analysis.
2.  **Load the Data**: Inside the notebook, you can use the pandas library to load the raw CSV files from the `data/raw/` directory.
3.  **Explore!**: You can then perform EDA to understand the data's characteristics, find patterns, and visualize it.

You can also import any useful functions from the `feature_engineering.py` script right into your notebook to see how they transform the data.

Step 1: Add src to the Python Path & Import Libraries
This is the setup phase. The first part tells your notebook where to find your project's custom Python files (.py), and the second part imports the standard data science libraries we'll need.

In [7]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# --- Add src to the Python Path ---
# This allows us to import our own project's modules
module_path = os.path.abspath(os.path.join('..', 'src'))
if module_path not in sys.path:
    sys.path.append(module_path)
    print(f"Added {module_path} to sys.path")

print("\nSetup Complete. Libraries imported.")


Setup Complete. Libraries imported.


Step 2: Fetch the Raw Data
Now, we'll use the function from the src/data/ directory to fetch the live data. We are assuming the function is named fetch_crypto_data in a file called ingestion.py. Remember to check the actual filenames in your project and adjust if needed!

Install the Binance Library (If Needed)
First, make sure you have the library installed. You can run this in your terminal or in a notebook cell (by putting a ! at the beginning).

pip install python-binance

In [8]:
import pandas as pd
from binance.client import Client

def fetch_crypto_data(symbol, start_date, interval=Client.KLINE_INTERVAL_1HOUR):
    """
    Fetches historical OHLCV data from Binance and returns it as a clean DataFrame.
    """
    print(f"Fetching {interval} data for {symbol} starting from {start_date}...")
    
    # 1. Connect to the Binance client (no API key needed for public data)
    client = Client()
    
    # 2. Fetch the raw data from the API
    # The API returns a list of lists, not a clean table
    klines = client.get_historical_klines(symbol=symbol, interval=interval, start_str=start_date)
    print(f"Found {len(klines)} records.")
    
    # 3. Define the column names based on the Binance API documentation
    columns = [
        'Open Time', 'Open', 'High', 'Low', 'Close', 'Volume', 
        'Close Time', 'Quote Asset Volume', 'Number of Trades', 
        'Taker Buy Base Asset Volume', 'Taker Buy Quote Asset Volume', 'Ignore'
    ]
    
    # 4. Convert the raw data into a pandas DataFrame
    df = pd.DataFrame(klines, columns=columns)
    
    # --- DATA CLEANING ---
    # 5. Convert the 'Open Time' from milliseconds to a readable datetime format
    df['Date'] = pd.to_datetime(df['Open Time'], unit='ms')
    
    # 6. Convert numeric columns from strings to numbers so we can do math
    numeric_cols = ['Open', 'High', 'Low', 'Close', 'Volume']
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce') # 'coerce' turns errors into NaN
        
    # 7. Keep only the columns we need and set the Date as the index
    df = df[['Date', 'Open', 'High', 'Low', 'Close', 'Volume']]
    df.set_index('Date', inplace=True)
    
    print("Data successfully cleaned and converted to DataFrame.")
    return df

Step 3: Use Your New Function!
Now, you can replace the old code in the "Fetch the Raw Data" cell with a simple call to your new function. It will feel so satisfying to see it run!

This will be your new "Fetch Data" cell:

In [9]:
# --- Use your new function to fetch the data ---
# Note: The Binance API often uses symbols like 'BTCUSDT' instead of 'BTC-USD'
symbol_to_fetch = 'BTCUSDT'
start_date_str = '1 Jan, 2024'

raw_df = fetch_crypto_data(symbol=symbol_to_fetch, start_date=start_date_str)

# --- Let's look at the result! ---
if not raw_df.empty:
    print("\n--- First 5 Rows of Fetched Data ---")
    display(raw_df.head())
else:
    print("Something went wrong, the DataFrame is empty.")

Fetching 1h data for BTCUSDT starting from 1 Jan, 2024...
Found 19239 records.
Data successfully cleaned and converted to DataFrame.

--- First 5 Rows of Fetched Data ---


,Open,High,Low,Close,Volume
Date,,,,,
2024-01-01 00:00:00,42283.58,42554.57,42261.02,42475.23,1271.68108
2024-01-01 01:00:00,42475.23,42775.00,42431.65,42613.56,1196.37856
2024-01-01 02:00:00,42613.57,42638.41,42500.00,42581.10,685.21980
2024-01-01 03:00:00,42581.09,42586.64,42230.08,42330.49,794.80391
2024-01-01 04:00:00,42330.50,42399.99,42209.46,42399.99,715.41760
